# V5 Training — Games + Mate Puzzles + Draws + Resignations

Trains from `engine.pt` (v3) on a mixed dataset that teaches:
- **Checkmate** — mate puzzles (`mateIn1-4`, `val=+1.0`, ~340k samples)
- **Draw recognition** — Hikaru 2024 draw positions (`val≈0`, ~60k samples)
- **Resignation** — lost positions (`val=-1.0`, ~5k samples)

**Setup before running:**
1. Kaggle Settings → Accelerator: **GPU T4 ×2**
2. Kaggle Settings → Add-ons → Secrets: add **HF_TOKEN** (write scope)
3. Add dataset: **omarcodeplay/gm-dataset** (contains `src/*.py`)

**Prerequisites (from local PC):**
- `Hikaru-Moves-2024.csv` uploaded to HF `chess-data`
- `resign_0000.npz` uploaded to HF `chess-data`

**Total runtime:** ~45 min (30 min training + 15 min setup/shard-building)

In [ ]:
# Cell 1 — Bootstrap: copy src files from read-only dataset mount
import os, shutil, pathlib

SRC_IN = '/kaggle/input/datasets/omarcodeplay/gm-dataset/src'
SRC_OUT = '/kaggle/working/src'
os.makedirs(SRC_OUT, exist_ok=True)

if not os.path.isdir(SRC_IN):
    import glob
    candidates = glob.glob('/kaggle/input/**/src', recursive=True)
    if candidates:
        SRC_IN = candidates[0]
        print(f'found src at: {SRC_IN}')
    else:
        raise SystemExit(f'Source not found at {SRC_IN} — add omarcodeplay/gm-dataset to the notebook')

copied = []
for f in pathlib.Path(SRC_IN).glob('*.py'):
    shutil.copy(f, pathlib.Path(SRC_OUT) / f.name)
    copied.append(f.name)

print(f'✅ Copied {len(copied)} files to {SRC_OUT}')
for f in sorted(copied):
    print(f'  {f}')

In [ ]:
# Cell 2 — Install dependencies
%%bash
pip install -q --upgrade python-chess huggingface_hub pandas numpy zstandard orjson

In [ ]:
# Cell 3 — Fix sync.py + engine.py + download Hikaru 2024
import os, sys
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_USER'] = 'OmarCodePlay'

import pathlib

# ── Patch sync.py: env precedence over Kaggle Secrets ──
sp = pathlib.Path('/kaggle/working/src/sync.py')
s = sp.read_text()
if 'if not TOKEN:' not in s:
    old = 'try:\n    from kaggle_secrets import UserSecretsClient\n    TOKEN = UserSecretsClient().get_secret("HF_TOKEN")\nexcept Exception:\n    TOKEN = os.environ.get("HF_TOKEN")'
    new = 'TOKEN = os.environ.get("HF_TOKEN")\nif not TOKEN:\n    try:\n        from kaggle_secrets import UserSecretsClient\n        TOKEN = UserSecretsClient().get_secret("HF_TOKEN")\n    except Exception:\n        TOKEN = None\nif not TOKEN:\n    raise RuntimeError("HF_TOKEN not found")'
    if old in s:
        sp.write_text(s.replace(old, new))
        print('✅ sync.py patched')
    else:
        print('⚠️ sync.py structure differs — writing fresh')
        sp.write_text('import os\nfrom huggingface_hub import HfApi, hf_hub_download\n\nTOKEN = os.environ.get("HF_TOKEN")\nif not TOKEN:\n    try:\n        from kaggle_secrets import UserSecretsClient\n        TOKEN = UserSecretsClient().get_secret("HF_TOKEN")\n    except Exception:\n        TOKEN = None\nif not TOKEN:\n    raise RuntimeError("HF_TOKEN not found")\n\nUSER = os.environ.get("HF_USER", "YOUR_HF_USERNAME")\nMODEL_REPO = f"{USER}/chess-ckpt"\nDATA_REPO = f"{USER}/chess-data"\napi = HfApi(token=TOKEN)\n\ndef init():\n    api.create_repo(MODEL_REPO, private=True, exist_ok=True)\n    api.create_repo(DATA_REPO, private=True, repo_type="dataset", exist_ok=True)\n\ndef push_ckpt(path, name=None):\n    api.upload_file(path_or_fileobj=path, path_in_repo=name or os.path.basename(path), repo_id=MODEL_REPO)\n\ndef pull_ckpt(name, d, force=False):\n    local = os.path.join(d, name)\n    if not force and os.path.exists(local):\n        return local\n    try:\n        return hf_hub_download(MODEL_REPO, name, local_dir=d, token=TOKEN, force_download=force)\n    except Exception:\n        return None\n\ndef push_data(path, name=None):\n    api.upload_file(path_or_fileobj=path, path_in_repo=name or os.path.basename(path), repo_id=DATA_REPO, repo_type="dataset")\n\ndef pull_data(name, d):\n    return hf_hub_download(DATA_REPO, name, repo_type="dataset", local_dir=d, token=TOKEN)\n\ndef list_data():\n    return [f.rfilename for f in api.list_repo_tree(DATA_REPO, repo_type="dataset")]\n')

# ── Fix engine.py zobrist_hash ──
ep = pathlib.Path('/kaggle/working/src/engine.py')
if ep.exists():
    esrc = ep.read_text()
    if 'board.zobrist_hash()' in esrc:
        esrc = esrc.replace('board.zobrist_hash()', 'chess.polyglot.zobrist_hash(board)')
        if 'import chess.polyglot' not in esrc:
            esrc = esrc.replace('import chess.syzygy', 'import chess.syzygy\nimport chess.polyglot')
        ep.write_text(esrc)
        print('✅ engine.py zobrist fixed')

# ── Reload sync ──
for k in list(sys.modules):
    if k == 'sync' or k.startswith('sync.'):
        del sys.modules[k]
sys.path.insert(0, '/kaggle/working/src')
import sync
print(f'✅ sync loaded, token={sync.TOKEN[:6]}...')

# ── Download Hikaru 2024 ──
os.makedirs('/kaggle/working/data', exist_ok=True)
csv_path = None
try:
    csv_path = sync.pull_data('Hikaru-Moves-2024.csv', '/kaggle/working/data')
    print(f'✅ from chess-data: {csv_path}')
except Exception as e:
    print(f'Not in chess-data ({e}), trying GM-games-with-eval...')
    from huggingface_hub import hf_hub_download
    csv_path = hf_hub_download('OmarCodePlay/GM-games-with-eval', 'Hikaru-Moves-2024.csv', repo_type='dataset', local_dir='/kaggle/working/data', token=os.environ['HF_TOKEN'])
    print(f'✅ from GM-games-with-eval: {csv_path}')

print(f'   size: {os.path.getsize(csv_path)/1e6:.1f} MB')
import pandas as pd
df = pd.read_csv(csv_path, low_memory=False)
print(f'   rows: {len(df):,}')
print(f'   result dist:')
print(df['result'].value_counts().to_string())

In [ ]:
# Cell 4 — Build mate shard (~340k samples, val=+1.0, fixes checkmate)
import os, sys, glob
sys.path.insert(0, '/kaggle/working/src')
import numpy as np, pandas as pd, chess
from encode import pack_board, move_index, save_npz

# Download Lichess puzzles if missing
parquet_files = sorted(glob.glob('/kaggle/working/default/train/*.parquet'))
if len(parquet_files) < 3:
    print('Downloading Lichess puzzles...')
    from huggingface_hub import hf_hub_download
    for i in ['0000', '0001', '0002']:
        p = hf_hub_download(repo_id='Lichess/chess-puzzles', filename=f'default/train/{i}.parquet', repo_type='dataset', revision='refs/convert/parquet', token=os.environ['HF_TOKEN'], local_dir='/kaggle/working')
        print(f'  {i}: {os.path.getsize(p)/1e6:.1f} MB')
    parquet_files = sorted(glob.glob('/kaggle/working/default/train/*.parquet'))

print(f'Loading {len(parquet_files)} parquet files...')
parts = [pd.read_parquet(f) for f in parquet_files]
df = pd.concat(parts, ignore_index=True)
print(f'Total puzzles: {len(df):,}')

# Theme token helper
def theme_tokens(themes):
    if themes is None: return []
    if isinstance(themes, str): return themes.split()
    try: return [str(t) for t in themes]
    except: return str(themes).split()

# Filter to mate-only puzzles
MATE_THEMES = {'mateIn1', 'mateIn2', 'mateIn3', 'mateIn4'}
def is_mate_puzzle(themes):
    return any(t in MATE_THEMES for t in theme_tokens(themes))

df['_is_mate'] = df['Themes'].apply(is_mate_puzzle)
mate_df = df[df['_is_mate']].copy()
print(f'Mate puzzles (all): {len(mate_df):,}')

mate_df = mate_df[(mate_df['Rating'] >= 1200) & (mate_df['Rating'] <= 2600)]
print(f'After rating:       {len(mate_df):,}')
mate_df = mate_df[mate_df['RatingDeviation'] <= 100]
print(f'After max RD:       {len(mate_df):,}')
mate_df = mate_df[mate_df['Popularity'].fillna(-100) >= 80]
print(f'After popularity:   {len(mate_df):,}')
mate_df = mate_df[mate_df['NbPlays'].fillna(0) >= 200]
print(f'After plays:        {len(mate_df):,}')

# Build samples
samples = []
seen_fens = set()
skipped = 0
for r in mate_df.to_dict('records'):
    fen = str(r.get('FEN', '') or '')
    moves_str = str(r.get('Moves', '') or '')
    if not fen or not moves_str: continue
    try:
        board = chess.Board(fen)
    except: continue
    moves = moves_str.split()
    if len(moves) < 2: continue
    try:
        setup = chess.Move.from_uci(moves[0])
        if setup not in board.legal_moves: continue
        board.push(setup)
    except: continue
    try:
        sol = chess.Move.from_uci(moves[1])
        if sol not in board.legal_moves: continue
    except: continue
    fen_before = board.fen()
    if fen_before in seen_fens: continue
    seen_fens.add(fen_before)
    black = board.turn == chess.BLACK
    try:
        bb, meta = pack_board(board)
    except:
        skipped += 1
        continue
    samples.append((
        bb, meta,
        np.array([move_index(sol, black), -1, -1, -1], dtype=np.int16),
        np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float16),
        1.0,
    ))

print(f'\nMate samples: {len(samples):,}  (skipped {skipped})')

os.makedirs('/kaggle/working/shards', exist_ok=True)
SHARD_SIZE = 300_000
mate_shard_names = []
for i in range(0, len(samples), SHARD_SIZE):
    chunk = samples[i:i + SHARD_SIZE]
    fn = f'mate_{i // SHARD_SIZE:04d}.npz'
    save_npz(chunk, f'/kaggle/working/shards/{fn}')
    print(f'  saved {fn}  ({len(chunk):,} rows)')
    mate_shard_names.append(fn)

In [ ]:
# Cell 5 — Build draws shard (~60k samples, val≈0, fixes draw recognition)
import os, sys, numpy as np, pandas as pd, chess, math
sys.path.insert(0, '/kaggle/working/src')
from encode import pack_board, move_index, save_npz

df = pd.read_csv('/kaggle/working/data/Hikaru-Moves-2024.csv', low_memory=False)
print(f'Total 2024: {len(df):,}')

draws = df[df['result'] == '1/2-1/2'].copy()
print(f'Draw rows:      {len(draws):,}')

if 'ply' in draws.columns:
    draws = draws[pd.to_numeric(draws['ply'], errors='coerce') >= 10]
    print(f'After book>=10: {len(draws):,}')

draws['eval_depth_before'] = pd.to_numeric(draws['eval_depth_before'], errors='coerce')
draws = draws[draws['eval_depth_before'] >= 12]
print(f'After depth>=12:{len(draws):,}')

def cp_to_v(cp):
    return 2.0 / (1.0 + math.exp(-0.00368208 * cp)) - 1.0

def parse_mate(x):
    if pd.isna(x): return None
    try: f = float(x)
    except: return None
    if f == 0: return None
    return 1.0 if f > 0 else -1.0

draws['eval_cp_before'] = pd.to_numeric(draws['eval_cp_before'], errors='coerce')
draws['eval_mate_before'] = pd.to_numeric(draws['eval_mate_before'], errors='coerce')

samples = []
seen_fens = set()
skipped = 0
for _, r in draws.iterrows():
    try:
        fen = r['fen']
        if fen in seen_fens: continue
        seen_fens.add(fen)
        b = chess.Board(fen)
        m = chess.Move.from_uci(str(r['move_uci']))
        if m not in b.legal_moves:
            skipped += 1
            continue
        black = b.turn == chess.BLACK
        mate_v = parse_mate(r['eval_mate_before'])
        if mate_v is not None:
            v = mate_v
        else:
            cp = r['eval_cp_before']
            if pd.isna(cp):
                skipped += 1
                continue
            v = cp_to_v(float(cp))
        if black: v = -v
        bb, meta = pack_board(b)
        samples.append((
            bb, meta,
            np.array([move_index(m, black), -1, -1, -1], dtype=np.int16),
            np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float16),
            float(v),
        ))
    except:
        skipped += 1
        continue

print(f'\nDraw samples: {len(samples):,}  (skipped {skipped})')
vals = np.array([s[4] for s in samples])
print(f'Value: min={vals.min():.3f} max={vals.max():.3f} mean={vals.mean():.3f}')
print(f'  |v| < 0.15:  {(np.abs(vals) < 0.15).mean()*100:5.1f}%')
print(f'  |v| < 0.30:  {(np.abs(vals) < 0.30).mean()*100:5.1f}%')

os.makedirs('/kaggle/working/shards', exist_ok=True)
draw_shard_names = []
SHARD_SIZE = 200_000
for i in range(0, len(samples), SHARD_SIZE):
    chunk = samples[i:i + SHARD_SIZE]
    fn = f'draws_{i // SHARD_SIZE:04d}.npz'
    save_npz(chunk, f'/kaggle/working/shards/{fn}')
    print(f'  saved {fn}  ({len(chunk):,} rows)')
    draw_shard_names.append(fn)

In [ ]:
# Cell 6 — Push new shards + rebuild index.json
import os, sys, json
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_USER'] = 'OmarCodePlay'
for k in list(sys.modules):
    if k == 'sync' or k.startswith('sync.'):
        del sys.modules[k]
sys.path.insert(0, '/kaggle/working/src')
import sync

# Push new mate + draws shards
new_shards = list(mate_shard_names) + list(draw_shard_names)
for fn in new_shards:
    local = f'/kaggle/working/shards/{fn}'
    if not os.path.exists(local):
        print(f'⚠️ {fn} not found locally — skipping')
        continue
    sync.push_data(local, fn)
    print(f'✅ pushed {fn}')

# Verify resign_0000.npz in chess-data
try:
    sync.pull_data('resign_0000.npz', '/tmp/verify')
    print('✅ resign_0000.npz in chess-data')
except Exception:
    print('❌ resign_0000.npz missing from chess-data — push it from local PC first')

# Rebuild index
idx = json.load(open(sync.pull_data('index.json', '/kaggle/working/shards')))
existing = list(idx['train'])
print(f'\nExisting: {len(existing)} shards')
for s in existing:
    print(f'  {s}')

for s in new_shards:
    if s not in existing:
        existing.append(s)
        print(f'  + {s}')

idx['train'] = existing
with open('/tmp/index.json', 'w') as f:
    json.dump(idx, f)
sync.push_data('/tmp/index.json', 'index.json')

print(f'\n✅ index.json updated — {len(existing)} shards total')

In [ ]:
# Cell 7 — Configure train.py for v5
import pathlib
p = pathlib.Path('/kaggle/working/src/train.py')
src = p.read_text()

# Fix map_location int-callable bug
src = src.replace('map_location=local_rank', 'map_location=f"cuda:{local_rank}"')

# Replace ALL existing run names -> v5
for old_run in ['base2', 'base3', 'base4', 'base5', 'base6', 'base7']:
    src = src.replace(f'run="{old_run}",', 'run="v5",')

# Ensure init_from is set
if 'init_from="engine.pt",' not in src:
    for old_init in ['init_from=None,', 'init_from="best_base3.pt",', 'init_from="best_base2.pt",']:
        src = src.replace(old_init, 'init_from="engine.pt",')

# LR
src = src.replace('lr=2e-3,', 'lr=4e-5,')
src = src.replace('lr=3e-5,', 'lr=4e-5,')
src = src.replace('lr=5e-5,', 'lr=4e-5,')

# Value weight
src = src.replace('v_w=2.0,', 'v_w=0.7,')
src = src.replace('v_w=0.8,', 'v_w=0.7,')
src = src.replace('v_w=0.5,', 'v_w=0.7,')

# Steps
src = src.replace('max_steps=12000,', 'max_steps=15000,')
src = src.replace('max_steps=10000,', 'max_steps=15000,')
src = src.replace('max_steps=None,', 'max_steps=15000,')

# Decay
src = src.replace('decay_start=9000,', 'decay_start=11000,')
src = src.replace('decay_start=7500,', 'decay_start=11000,')
src = src.replace('decay_start=None,', 'decay_start=11000,')
src = src.replace('decay_steps=3000,', 'decay_steps=4000,')
src = src.replace('decay_steps=2500,', 'decay_steps=4000,')
src = src.replace('decay_steps=100_000,', 'decay_steps=4000,')

# Ensure patience + tolerance exist in C dict
if 'patience' not in src:
    src = src.replace('max_steps=15000,', 'max_steps=15000,\n    patience=3,\n    valmse_tolerance=0.0005,')

p.write_text(src)
print('✅ train.py configured for v5\n')

!grep -nE '^    run=|^    init_from=|^    lr=|^    v_w=|^    max_steps=|^    decay_start=|^    patience=' /kaggle/working/src/train.py

In [ ]:
# Cell 8 — Verify + patch pol_loss masking (value-only samples)
import subprocess
r = subprocess.run(['grep', '-c', 'has_policy', '/kaggle/working/src/train.py'], capture_output=True, text=True)
count = int(r.stdout.strip() or '0')

if count >= 2:
    print(f'✅ pol_loss already masked ({count} occurrences)')
else:
    print(f'⚠️ pol_loss NOT masked ({count} occurrences) — applying patch...')
    import pathlib
    p = pathlib.Path('/kaggle/working/src/train.py')
    src = p.read_text()
    old = 'def pol_loss(lg, mvk, wk, smooth):\n    lp = F.log_softmax(lg.float(), 1)\n    w = wk.float()\n    w = w / w.sum(1, keepdim=True).clamp_min(1e-6)\n    nll = -(w * lp.gather(1, mvk.long().clamp_min(0))).sum(1).mean()\n    return (1 - smooth) * nll + smooth * (-lp.mean(1)).mean()'
    new = 'def pol_loss(lg, mvk, wk, smooth):\n    has_policy = (wk.sum(dim=1) > 0)\n    if not has_policy.any():\n        return torch.zeros((), device=lg.device, dtype=torch.float32)\n    lg_m = lg[has_policy]\n    mvk_m = mvk[has_policy]\n    wk_m = wk[has_policy]\n    lp = F.log_softmax(lg_m.float(), 1)\n    w = wk_m.float()\n    w = w / w.sum(1, keepdim=True).clamp_min(1e-6)\n    nll = -(w * lp.gather(1, mvk_m.long().clamp_min(0))).sum(1).mean()\n    return (1 - smooth) * nll + smooth * (-lp.mean(1)).mean()'
    if old in src:
        p.write_text(src.replace(old, new))
        print('✅ pol_loss patched')
    else:
        print('❌ pol_loss pattern not found — check train.py structure')

In [ ]:
# Cell 9 — Launch v5 training (~25-35 min on 2x T4)
import os, sys, json
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_USER'] = 'OmarCodePlay'
for k in list(sys.modules):
    if k == 'sync' or k.startswith('sync.'):
        del sys.modules[k]
sys.path.insert(0, '/kaggle/working/src')
import sync

# Pre-flight check
idx = json.load(open(sync.pull_data('index.json', '/kaggle/working/shards')))
print(f'Train shards ({len(idx["train"])}):')
for s in idx['train']:
    print(f'  {s}')
print(f'Val:  {idx["val"]}')
print(f'Test: {idx["test"]}\n')

import subprocess
r = subprocess.run(['grep', '-E', 'run=|init_from=', '/kaggle/working/src/train.py'], capture_output=True, text=True)
print(r.stdout)
print('=' * 60)

# Launch
!cd /kaggle/working && torchrun --standalone --nproc_per_node=2 src/train.py 2>&1 | tee -a /kaggle/working/train_v5.log

In [ ]:
# Cell 10 — Test v5: checkmate + draw recognition + lost-position recognition
import chess
import pathlib
import sys
sys.path.insert(0, '/kaggle/working/src')

# Ensure engine.py is importable and zobrist is fixed
ep = pathlib.Path('/kaggle/working/src/engine.py')
if ep.exists():
    esrc = ep.read_text()
    if 'board.zobrist_hash()' in esrc:
        esrc = esrc.replace('board.zobrist_hash()', 'chess.polyglot.zobrist_hash(board)')
        if 'import chess.polyglot' not in esrc:
            esrc = esrc.replace('import chess.syzygy', 'import chess.syzygy\nimport chess.polyglot')
        ep.write_text(esrc)

from engine import Engine

print('=' * 60)
print('TEST 1: Mate-in-1 (back rank) — v5 should play mate')
print('=' * 60)
fen = '6k1/5ppp/8/8/8/8/5PPP/R5K1 w - - 0 1'
board = chess.Board(fen)
v5 = Engine('/kaggle/working/ckpt/best_v5.pt')
v3 = Engine('/kaggle/working/ckpt/engine.pt')
mv5, _ = v5.search(board, float('inf'), float('inf'), 500, None)
mv3, _ = v3.search(board, float('inf'), float('inf'), 500, None)
bc = board.copy()
bc.push(mv5)
v5_mate = bc.is_checkmate()
print(f'v5 plays: {mv5.uci()} -> "{"MATE ✅" if v5_mate else "not mate ❌"}"')
print(f'v3 plays: {mv3.uci()}')

print()
print('=' * 60)
print('TEST 2: Draw recognition (K+R vs K+R — should be near 0)')
print('=' * 60)
draw_fen = '8/8/4k3/8/8/4K3/8/R6r w - - 0 1'
board_d = chess.Board(draw_fen)
_, r5 = v5.search(board_d, float('inf'), float('inf'), 200, None)
_, r3 = v3.search(board_d, float('inf'), float('inf'), 200, None)
val5 = r5.w / r5.n if r5 and r5.n else 0
val3 = r3.w / r3.n if r3 and r3.n else 0
print(f'v5 value: {val5:+.3f}  "{"✅ near 0" if abs(val5) < 0.3 else "⚠️ overconfident"}"')
print(f'v3 value: {val3:+.3f}  "{"✅ near 0" if abs(val3) < 0.3 else "⚠️ overconfident"}"')

print()
print('=' * 60)
print('TEST 3: Lost position (opponent down a queen — should be near +1)')
print('=' * 60)
won_fen = 'rnb1kbnr/pppp1ppp/8/8/8/8/PPPP1PPP/RNBQKBNR w KQkq - 0 1'
board_w = chess.Board(won_fen)
_, r5w = v5.search(board_w, float('inf'), float('inf'), 200, None)
_, r3w = v3.search(board_w, float('inf'), float('inf'), 200, None)
val5w = r5w.w / r5w.n if r5w and r5w.n else 0
val3w = r3w.w / r3w.n if r3w and r3w.n else 0
print(f'v5 value: {val5w:+.3f}  "{"✅ near +1" if val5w > 0.5 else "⚠️ weak"}"')
print(f'v3 value: {val3w:+.3f}  "{"✅ near +1" if val3w > 0.5 else "⚠️ weak"}"')